## Setup & Initial Data Load

Loading a single month (January 2025) first to understand the raw schema before deciding
which columns are worth carrying through the full pipeline. The BTS Reporting Carrier
On-Time Performance dataset ships with 110 columns per month, many of which are
legacy/rarely-populated fields (e.g., secondary/tertiary diversion airport details).

In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/flights_2025_01.csv')
print(df.shape)
df.head()

/var/folders/w7/9yvpw7ls4zz1270ckxmfr0kr0000gn/T/ipykernel_73856/1767233799.py:3: DtypeWarning: Columns (76,77,84) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/flights_2025_01.csv')


(539747, 110)


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2025,1,1,1,3,2025-01-01,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,19805,AA,N110AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,19805,AA,N106NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,19805,AA,N117AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
keywords = ['Div', 'Carrier', 'Weather', 'NAS', 'Security', 'LateAircraft', 'Delay', 'Cancel', 'Diverted']
for kw in keywords:
    matches = [c for c in df.columns if kw in c]
    print(f'{kw}: {len(matches)} columns')
    print(matches)
    print()

Div: 46 columns
['Diverted', 'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay', 'DivDistance', 'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn', 'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum', 'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn', 'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum', 'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn', 'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum', 'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn', 'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum']

Carrier: 1 columns
['CarrierDelay']

Weather: 1 columns
['WeatherDelay']

NAS: 1 columns
['NASDelay']

Security: 1 columns
['SecurityDelay']

LateAircraft: 1 columns
['LateAircraftDelay'

## Column Selection

The raw dataset ships with 110 columns, but 46 of them (`Div1`–`Div5` fields) only
populate for flights diverted to a secondary airport — a rare edge case that adds
significant width for minimal analytical value. Trimming down to ~35 columns focused
on flight identity, origin/destination, departure/arrival performance, and delay-cause
breakdown, which covers the actual business questions this project is answering
(on-time reliability by route/carrier/season, and what's driving

In [10]:
keep_cols = [
    # Time & flight identity
    'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline',

    # Airports
    'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState',

    # Departure performance
    'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepTimeBlk',

    # Arrival performance
    'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrTimeBlk',

    # Flight characteristics
    'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Distance', 'DistanceGroup',

    # Delay cause breakdown
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
]

df_trimmed = df[keep_cols]
print(df_trimmed.shape)
df_trimmed.head()

(539747, 39)


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,OriginCityName,...,CRSElapsedTime,ActualElapsedTime,AirTime,Distance,DistanceGroup,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2025,1,1,1,3,2025-01-01,AA,1,JFK,"New York, NY",...,381.0,377.0,345.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,1,JFK,"New York, NY",...,381.0,390.0,353.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,1,JFK,"New York, NY",...,381.0,371.0,347.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,1,JFK,"New York, NY",...,386.0,383.0,349.0,2475.0,10,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,1,JFK,"New York, NY",...,381.0,378.0,347.0,2475.0,10,NaN,NaN,NaN,NaN,NaN


## Data Quality Check — Nulls & Types

Checking `.info()` for dtypes and `.isnull().sum()` for missingness patterns. Some nulls
are expected by design, not data quality issues: delay-cause columns (`CarrierDelay`,
`WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`) only populate when a
flight was actually delayed 15+ minutes, and time/delay fields like `DepTime`/`ArrDelay`
will be null for cancelled flights since they never departed or arrived.

In [11]:
df_trimmed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 539747 entries, 0 to 539746
Data columns (total 39 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   Year                             539747 non-null  int64  
 1   Quarter                          539747 non-null  int64  
 2   Month                            539747 non-null  int64  
 3   DayofMonth                       539747 non-null  int64  
 4   DayOfWeek                        539747 non-null  int64  
 5   FlightDate                       539747 non-null  object 
 6   Reporting_Airline                539747 non-null  object 
 7   Flight_Number_Reporting_Airline  539747 non-null  int64  
 8   Origin                           539747 non-null  object 
 9   OriginCityName                   539747 non-null  object 
 10  OriginState                      539747 non-null  object 
 11  Dest                             539747 non-null  object 
 12  De

In [14]:
null_counts = df_trimmed.isnull().sum()
null_pct = (null_counts / len(df_trimmed) * 100).round(2)

null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary = null_summary[null_summary['null_count'] > 0].sort_values('null_pct', ascending=False)
null_summary

,null_count,null_pct
CancellationCode,523435,96.98
CarrierDelay,441617,81.82
WeatherDelay,441617,81.82
NASDelay,441617,81.82
SecurityDelay,441617,81.82
LateAircraftDelay,441617,81.82
ArrDelay,17478,3.24
ArrDelayMinutes,17478,3.24
ArrDel15,17478,3.24
ActualElapsedTime,17478,3.24


### Findings — Null Patterns Are Structural, Not Missing Data

- **`CancellationCode`** (97.0% null) — only populated for cancelled flights (~3% of the dataset).
- **Delay-cause columns** (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`,
  `LateAircraftDelay` — all 81.8% null) — BTS only reports a delay cause breakdown when a
  flight arrives 15+ minutes late (`ArrDel15 = 1`). Identical null counts across all five
  confirm they're populated as a set.
- **Arrival-side fields** (~3.2% null) and **departure-side fields** (~2.95% null) — the
  small gap reflects that a flight can be cancelled pre-departure (no `DepTime`/`ArrTime`)
  or diverted after departure (`DepTime` present, `ArrTime` missing).

**Cleaning approach going forward:**
- Delay-cause nulls will be filled with 0 (not applicable), not dropped or imputed.
- Cancelled/diverted flights will be retained as their own category rather than removed —
  cancellation/diversion rate is itself a reliability metric worth surfacing per route/carrier.

## Loading & Combining All 12 Months

Applying the same column selection to all 12 monthly files and concatenating into a
single DataFrame before loading into Postgres.

In [15]:
import glob

file_paths = sorted(glob.glob('../data/raw/flights_2025_*.csv'))
print(f'Found {len(file_paths)} files')

dfs = []
for path in file_paths:
    monthly_df = pd.read_csv(path, usecols=keep_cols, low_memory=False)
    dfs.append(monthly_df)
    print(f'{path}: {monthly_df.shape}')

df_all = pd.concat(dfs, ignore_index=True)
print(f'Combined shape: {df_all.shape}')

Found 12 files
../data/raw/flights_2025_01.csv: (539747, 39)
../data/raw/flights_2025_02.csv: (504884, 39)
../data/raw/flights_2025_03.csv: (600872, 39)
../data/raw/flights_2025_04.csv: (583950, 39)
../data/raw/flights_2025_05.csv: (605648, 39)
../data/raw/flights_2025_06.csv: (611575, 39)
../data/raw/flights_2025_07.csv: (631428, 39)
../data/raw/flights_2025_08.csv: (602378, 39)
../data/raw/flights_2025_09.csv: (562439, 39)
../data/raw/flights_2025_10.csv: (605844, 39)
../data/raw/flights_2025_11.csv: (570550, 39)
../data/raw/flights_2025_12.csv: (582304, 39)
Combined shape: (7001619, 39)


### Result

All 12 months loaded and combined successfully: **7,001,619 flights** across 2025,
39 columns, consistent with expected US domestic flight volume for a full year.
This combined DataFrame (`df_all`) is now ready to be loaded into Postgres.

## Schema Design — Star Schema

Chose a star schema over a flat table to properly model this as a data warehouse:
- `dim_date`, `dim_airport`, `dim_carrier` — dimension tables
- `fact_flights` — fact table, with `dim_airport` referenced twice (origin and destination),
  a "role-playing dimension"

This mirrors how flight/booking data would realistically be modeled in a production
analytics environment, and sets up the fact table for efficient aggregation queries
(rolling on-time rates, carrier scorecards, route-level reliability) via SQL joins
rather than repeated wide-table scans.